# 03 · Transformer 去雾块

RescaleNorm + 窗口多头注意力 + 复原友好 ReLU；验证前向形状。

In [ ]:
import torch
import torch.nn as nn


class RescaleNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(1, dim, 1, 1))
        self.bias = nn.Parameter(torch.zeros(1, dim, 1, 1))
        self.meta1 = nn.Conv2d(1, dim, 1)
        self.meta2 = nn.Conv2d(1, dim, 1)

    def forward(self, x):
        mean = x.mean(dim=(1, 2, 3), keepdim=True)
        std = torch.sqrt(((x - mean) ** 2).mean(dim=(1, 2, 3), keepdim=True) + self.eps)
        xn = (x - mean) / std
        return xn * self.weight + self.bias + self.meta1(std) + self.meta2(mean)


class WindowAttention(nn.Module):
    def __init__(self, dim, window_size, num_heads):
        super().__init__()
        self.dim, self.num_heads = dim, num_heads
        self.window_size = window_size
        self.scale = (dim // num_heads) ** -0.5
        self.qkv = nn.Conv2d(dim, dim * 3, 1)
        self.proj = nn.Conv2d(dim, dim, 1)

    def forward(self, x):
        B, C, H, W = x.shape
        ws = self.window_size
        x = x.view(B, C, H // ws, ws, W // ws, ws)
        x = x.permute(0, 2, 4, 3, 5, 1).contiguous().view(-1, ws * ws, C)
        qkv = self.qkv(x.transpose(1, 2).unsqueeze(-1)).squeeze(-1)
        qkv = qkv.view(-1, ws * ws, 3, self.num_heads, C // self.num_heads)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = torch.softmax((q @ k.transpose(-2, -1)) * self.scale, dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(-1, ws * ws, C)
        out = out.view(B, H // ws, W // ws, ws, ws, C).permute(0, 5, 1, 3, 2, 4)
        return self.proj(out.contiguous().view(B, C, H, W))


class DehazFormerBlock(nn.Module):
    def __init__(self, dim, num_heads, window_size, mlp_ratio=2.0):
        super().__init__()
        self.norm1 = RescaleNorm(dim)
        self.attn = WindowAttention(dim, window_size, num_heads)
        self.norm2 = RescaleNorm(dim)
        hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Conv2d(dim, hidden, 1), nn.ReLU(inplace=True),
            nn.Conv2d(hidden, dim, 1))

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class DehazeTransformer(nn.Module):
    def __init__(self, dim=32, num_heads=4, window_size=8, depth=2):
        super().__init__()
        self.stem = nn.Conv2d(3, dim, 3, padding=1)
        self.blocks = nn.Sequential(*[
            DehazFormerBlock(dim, num_heads, window_size) for _ in range(depth)])
        self.head = nn.Conv2d(dim, 3, 3, padding=1)

    def forward(self, x):
        return x + self.head(self.blocks(self.stem(x)))


In [ ]:
model = DehazeTransformer(dim=32, num_heads=4, window_size=8, depth=2)
x = torch.randn(1, 3, 256, 256)
y = model(x)
print("输入:", x.shape, "输出:", y.shape)
print("参数量: %.2f M" % (sum(p.numel() for p in model.parameters()) / 1e6))
